# Round-7 #6 -- Held-out-CONTINGENCY generalization test

Distinguishes **physics-generalization** from **branch-identity memorization**. We hold out
20% of branches (set H, 9 of 48, fixed seed 7). Every sample that trips at least one held-out
branch is the generalization test set (8,465 samples) -- those contingencies are **never seen
in training**. The trainable pool (all tripped branches seen, incl. base cases) is split
sample-wise into train / val / in-distribution test.

* Operating points are shared across train and held-out test **on purpose** -- isolates the
  branch-identity variable. OP-generalization is covered by the main grouped-by-OP split; the
  two are complementary factorial cells.
* In-distribution (ID) test = sample-slice of the trainable pool -> the ID-vs-held-out gap is a
  clean single-variable (branch novelty) measurement from the **same** trained weights.
* Model selection uses ID val only; the held-out set is touched **once**, at final eval.
* These numbers are **NOT** comparable to the deployed 98.12% (different split protocol / train size).

Trains the exact deployed config: masked coupled (alpha=1), 150 epochs, 5 seeds, best-val-at-0-fs
checkpoint. Runtime -> GPU, Run all. Saves `heldout_results.json`.

In [ ]:
from google.colab import drive
import glob, os, sys
drive.mount('/content/drive')
c = glob.glob('/content/drive/MyDrive/**/smart_load_shield_boost', recursive=True)
FOLDER = c[0] if c else '/content/drive/MyDrive/smart_load_shield_boost'
sys.path.insert(0, FOLDER)
need = ['contingency_data.npz', 'heldout_split.npz', 'boost_core.py']
missing = [f for f in need if not os.path.exists(os.path.join(FOLDER, f))]
assert not missing, 'missing in ' + FOLDER + ': ' + str(missing)
assert 'base_mask' in open(os.path.join(FOLDER, 'boost_core.py')).read(), \
    'boost_core.py in FOLDER is the OLD unmasked version -- replace with round-6 masked'
print('FOLDER =', FOLDER, '| masked boost_core OK')

In [ ]:
import numpy as np, torch, json
import boost_core as B
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device', DEV)

# build_data reads idx_tr/idx_va/idx_te (= ID test) + train-only norm from heldout_split.npz
d = B.build_data(os.path.join(FOLDER, 'contingency_data.npz'),
                 os.path.join(FOLDER, 'heldout_split.npz'), DEV)
S = np.load(os.path.join(FOLDER, 'heldout_split.npz'))
Draw = np.load(os.path.join(FOLDER, 'contingency_data.npz'))
H = S['H']; kcnt = S['k']
idx_heldout = torch.tensor(np.load(os.path.join(FOLDER, 'heldout_split.npz'))['idx_heldout'].astype('int64')).to(DEV)
tripped = (Draw['in_service'] == 0)             # (M, n_branch) for per-branch breakdown
print('H =', H.tolist(), '| train/val/IDtest/heldout =',
      len(d['idx_tr']), len(d['idx_va']), len(d['idx_te']), len(idx_heldout))

In [ ]:
# ---- full metric helper: risk acc / macro-F1 / per-class recall / false-safe / Vmin-MAE ----
@torch.no_grad()
def get_preds(net, idx):
    net.eval(); P, VM = [], []
    for s in range(0, len(idx), 4096):
        b = idx[s:s+4096]
        r, vm, vu, dv, vb = net(d['NF'][b], d['DENSE'][b], d['ADJ'][b])
        P.append(r.argmax(1)); VM.append(vm)
    P = torch.cat(P).cpu().numpy(); VM = torch.cat(VM).cpu().numpy()
    R = d['Tr'][idx].cpu().numpy(); conv = (d['Td'][idx] == 0).cpu().numpy()
    Vt = d['Tv'][idx].cpu().numpy()
    return P, VM, R, conv, Vt

def metrics(P, VM, R, conv, Vt):
    cm = np.zeros((3, 3), int)
    for t, p in zip(R, P): cm[t, p] += 1
    acc = float((P == R).mean())
    recall = [float(cm[c, c] / max(1, cm[c].sum())) for c in range(3)]
    f1 = []
    for k in range(3):
        tp = cm[k, k]; fp = cm[:, k].sum() - tp; fn = cm[k, :].sum() - tp
        pr = tp / max(1, tp + fp); rc = tp / max(1, tp + fn); f1.append(2*pr*rc/max(1e-9, pr+rc))
    macro_f1 = float(np.mean(f1))
    fs = float(cm[2, 0] / max(1, cm[2].sum()))          # Unstable predicted Stable
    fs_n = int(cm[2, 0])
    m = conv
    vmae = float(np.abs(Vt[m] - VM[m]).mean()) if m.any() else float('nan')
    return dict(n=int(len(R)), acc=acc, macro_f1=macro_f1, recall_S=recall[0], recall_M=recall[1],
                recall_U=recall[2], false_safe=fs, false_safe_n=fs_n, n_unstable=int(cm[2].sum()),
                vmin_mae=vmae, cm=cm.tolist())

In [ ]:
# ---- train the deployed config x5 seeds; keep best val-at-fs0 checkpoint (deploy protocol) ----
SEEDS = [0, 1, 2, 3, 4]
per_seed = []
best_val, best_state, best_seed = -1.0, None, None
for sd in SEEDS:
    te, model = B.train_eval(d, seed=sd, alpha=1.0, branched=False, stack=False, vbus=False,
                             epochs=150, amp=(DEV == 'cuda'))
    idm = metrics(*get_preds(model, d['idx_te']))          # in-distribution test
    hom = metrics(*get_preds(model, idx_heldout))          # held-out-branch test
    val = B.evaluate(model, d, d['idx_va'])
    per_seed.append(dict(seed=sd, val_acc=val['acc'], val_fs=val['false_safe'],
                         id=idm, heldout=hom))
    print('seed%d  val %.2f%%(fs%.2f)  ID acc %.2f%% fs %.2f%%  | HELD-OUT acc %.2f%% mF1 %.3f fs %.3f%% (n=%d) vmae %.4f'
          % (sd, val['acc']*100, val['false_safe']*100, idm['acc']*100, idm['false_safe']*100,
             hom['acc']*100, hom['macro_f1'], hom['false_safe']*100, hom['false_safe_n'], hom['vmin_mae']))
    if val['false_safe'] == 0.0 and val['acc'] > best_val:
        best_val = val['acc']; best_state = {k: v.clone() for k, v in model.state_dict().items()}; best_seed = sd
assert best_state is not None, 'no seed reached 0 false-safe on val'
print('\nselected checkpoint: seed %d (val acc %.2f%%)' % (best_seed, best_val*100))

In [ ]:
# ---- mean +/- std across seeds, both test sets ----
def agg(key, sub):
    v = np.array([p[key][sub] for p in per_seed], float)
    return float(v.mean()), float(v.std())
summary = {}
for sub in ['id', 'heldout']:
    summary[sub] = {}
    for key in ['acc', 'macro_f1', 'recall_U', 'false_safe', 'vmin_mae']:
        mu, sdv = agg(sub, key); summary[sub][key] = [mu, sdv]
    summary[sub]['false_safe_n_per_seed'] = [p[sub]['false_safe_n'] for p in per_seed]
print('=== mean +/- std over 5 seeds ===')
for sub in ['id', 'heldout']:
    s = summary[sub]
    print('%-8s acc %.2f+-%.2f%%  mF1 %.3f+-%.3f  U-recall %.2f+-%.2f%%  fs %.3f+-%.3f%%  vmae %.4f+-%.4f'
          % (sub, s['acc'][0]*100, s['acc'][1]*100, s['macro_f1'][0], s['macro_f1'][1],
             s['recall_U'][0]*100, s['recall_U'][1]*100, s['false_safe'][0]*100, s['false_safe'][1]*100,
             s['vmin_mae'][0], s['vmin_mae'][1]))
print('held-out false-safe counts per seed:', summary['heldout']['false_safe_n_per_seed'])
print('ID gap: held-out acc is', round((summary['id']['acc'][0]-summary['heldout']['acc'][0])*100, 2), 'pp below ID')

In [ ]:
# ---- per-held-out-branch table on the SELECTED checkpoint (kills "easy branches") ----
m = B.CSGNNv2(node_dim=4, edge_dim=3, hidden=128, heads=4, branched=False, stack=False, vbus=False).to(DEV)
m.load_state_dict(best_state); m.set_base_mask(d['BASE']); m.eval()
hidx = idx_heldout.cpu().numpy()
P, VM, R, conv, Vt = get_preds(m, idx_heldout)
per_branch = []
for j in H:
    sub = tripped[hidx, j]                        # held-out samples that trip branch j
    if sub.sum() == 0: continue
    bm = metrics(P[sub], VM[sub], R[sub], conv[sub], Vt[sub])
    a, b_ = Draw['edge_pairs'][j]; pair = (int(Draw['bus_ids'][a]), int(Draw['bus_ids'][b_]))
    bm.update(branch=int(j), pair=pair); per_branch.append(bm)
    print('branch %2d %s  n=%4d  acc %.1f%%  U-rec %.1f%%  fs %d/%d  vmae %.4f'
          % (j, pair, bm['n'], bm['acc']*100, bm['recall_U']*100, bm['false_safe_n'], bm['n_unstable'], bm['vmin_mae']))

# ---- pure N-1 held-out breakout (single tripped branch, held-out) = purest signal ----
n1mask = (kcnt[hidx] == 1)
n1 = metrics(P[n1mask], VM[n1mask], R[n1mask], conv[n1mask], Vt[n1mask])
print('\nPURE N-1 held-out (n=%d): acc %.2f%%  mF1 %.3f  U-recall %.2f%%  fs %d/%d  vmae %.4f'
      % (n1['n'], n1['acc']*100, n1['macro_f1'], n1['recall_U']*100, n1['false_safe_n'], n1['n_unstable'], n1['vmin_mae']))
# all-held-out multi-trip cell (k>=2 and every tripped branch is held-out): hardest
allheld = np.array([kcnt[i] >= 2 and tripped[i, H].sum() == kcnt[i] for i in hidx])
if allheld.sum():
    ah = metrics(P[allheld], VM[allheld], R[allheld], conv[allheld], Vt[allheld])
    print('ALL-held-out multi-trip (k>=2, n=%d): acc %.2f%%  U-recall %.2f%%  fs %d/%d'
          % (ah['n'], ah['acc']*100, ah['recall_U']*100, ah['false_safe_n'], ah['n_unstable']))
else:
    ah = None

In [ ]:
# ---- reviewer's "held-out N-2 pairs" clause: N-2 contingencies where NEITHER branch was seen ----
# (subset of the held-out test; strictly harder than pure N-1 -- two simultaneous novel trips)
n2both = np.array([kcnt[i] == 2 and tripped[i, H].sum() == 2 for i in hidx])
if n2both.sum():
    npm = metrics(P[n2both], VM[n2both], R[n2both], conv[n2both], Vt[n2both])
    print('Held-out N-2 (both branches unseen) n=%d: acc %.2f%%  U-recall %.2f%%  fs %d/%d  vmae %.4f'
          % (npm['n'], npm['acc']*100, npm['recall_U']*100, npm['false_safe_n'], npm['n_unstable'], npm['vmin_mae']))
else:
    npm = None; print('no both-held-out N-2 samples')

In [ ]:
# ---- save everything the paper needs ----
out = dict(
    design=dict(hold_seed=int(S['hold_seed']), split_seed=int(S['split_seed']),
                held_out_branches=H.tolist(), n_branch=int(Draw['n_branch']),
                n_train=len(d['idx_tr']), n_val=len(d['idx_va']),
                n_id_test=len(d['idx_te']), n_heldout=len(idx_heldout)),
    per_seed=per_seed, summary=summary, selected_seed=best_seed,
    per_branch=per_branch, pure_n1=n1, all_heldout_multi=ah, heldout_n2_pairs=npm)
with open(os.path.join(FOLDER, 'heldout_results.json'), 'w') as f:
    json.dump(out, f, indent=2)
print('saved heldout_results.json to', FOLDER)
print('DOWNLOAD this file back into redesign/ for the paper.')